# Modularnost

## Moduli

In [1]:
module Datum = struct
    type t = { leto : int; mesec : int; dan : int }

    let je_prestopno leto =
      (leto mod 4 = 0 && leto mod 100 <> 0) || leto mod 400 = 0

    let dolzina_meseca leto =
      function
      | 4 | 6 | 9 | 11 -> 30
      | 2 -> if je_prestopno leto then 29 else 28
      | _ -> 31

    let je_veljaven { leto; mesec; dan } =
      let veljaven_dan = 1 <= dan && dan <= dolzina_meseca leto mesec
      and veljaven_mesec = 1 <= mesec && mesec <= 12
      in
      veljaven_dan && veljaven_mesec

    let naredi leto mesec dan =
      let datum = { leto; mesec; dan } in
      if je_veljaven datum then Some datum else None

    let to_string { leto; mesec; dan } =
      Format.sprintf "%04d-%02d-%02d" leto mesec dan
  end

module Datum :
  sig
    type t = { leto : int; mesec : int; dan : int; }
    val je_prestopno : int -> bool
    val dolzina_meseca : int -> int -> int
    val je_veljaven : t -> bool
    val naredi : int -> int -> int -> t option
    val to_string : t -> string
  end


In [ ]:
open Datum

In [6]:
let open Datum in
{ leto = 1991; mesec = 6; dan = 25 }

- : Datum.t = {Datum.leto = 1991; mesec = 6; dan = 25}


In [ ]:
Datum.dolzina_meseca 1991 6

In [ ]:
let open Datum in
  je_veljaven { leto = 1991; mesec = 6; dan = 31 }

In [ ]:
Datum.(je_veljaven { leto = 1991; mesec = 6; dan = 31 })

## Signature

In [ ]:
module type DATUM = sig
    type t = { leto : int; mesec : int; dan : int; }
    val naredi : int -> int -> int -> t option
    val to_string : t -> string
  end
    

module type DATUM =
  sig
    type t = { leto : int; mesec : int; dan : int; }
    val je_prestopno : int -> bool
    val dolzina_meseca : int -> int -> int
    val je_veljaven : t -> bool
    val naredi : int -> int -> int -> t option
    val to_string : t -> string
  end


## Preverjanje skladnosti implementacije

In [13]:
module Datum : DATUM = struct
    type t = { leto : int; mesec : int; dan : int }

    let je_prestopno leto =
      (leto mod 4 = 0 && leto mod 100 <> 0) || leto mod 400 = 0

    let dolzina_meseca leto =
      function
      | 4 | 6 | 9 | 11 -> 30
      | 2 -> if je_prestopno leto then 29 else 28
      | _ -> 31

    let je_veljaven { leto; mesec; dan } =
      let veljaven_dan = 1 <= dan && dan <= dolzina_meseca leto mesec
      and veljaven_mesec = 1 <= mesec && mesec <= 12
      in
      veljaven_dan && veljaven_mesec

    let naredi leto mesec dan =
      let datum = { leto; mesec; dan } in
      if je_veljaven datum then Some datum else None

    let to_iso_string { leto; mesec; dan } =
        Format.sprintf "%04d-%02d-%02d" leto mesec dan

    let to_string { leto; mesec; dan } =
        Format.sprintf "%02d. %02d. %04d" dan mesec leto
  end

module Datum : DATUM


In [14]:
Datum.to_string { leto = 1991; mesec = 6; dan = 25 }

- : string = "25. 06. 1991"


In [15]:
Datum.to_iso_string { leto = 1991; mesec = 6; dan = 25 }

error: compile_error

In [21]:
module VednoVeljavenDatum : DATUM = struct
    type t = { dan : int; mesec : int; leto : int }

    let je_prestopno _ = false

    let dolzina_meseca _ _ = 31

    let je_veljaven _ = true

    let naredi leto mesec dan = Some { leto; mesec; dan }

    let to_string { leto; mesec; dan } =
      Format.sprintf "%04d-%02d-%02d" leto mesec dan
  end

error: compile_error

In [ ]:
module VednoVeljavenDatum : DATUM = struct
    type t = { leto : int; mesec : int; dan : int }

    let je_prestopno _ = false

    let dolzina_meseca _ _ = 31

    let je_veljaven _ = true

    let naredi leto mesec dan = Some { leto; mesec; dan }
  end

In [ ]:
module VednoVeljavenDatum : DATUM = struct
    type t = { leto : int; mesec : int; dan : int }

    let je_prestopno _ = false

    let dolzina_meseca _ _ = 31

    let je_veljaven _ = true

    let naredi (leto, mesec, dan) = Some { leto; mesec; dan }

    let to_string { leto; mesec; dan } =
      Format.sprintf "%04d-%02d-%02d" leto mesec dan
  end

## Skrivanje implementacije

In [17]:
module type DATUM =
    sig
      type t = { leto : int; mesec : int; dan : int }
      val naredi : int -> int -> int -> t option
      val to_string : t -> string
    end

module type DATUM =
  sig
    type t = { leto : int; mesec : int; dan : int; }
    val naredi : int -> int -> int -> t option
    val to_string : t -> string
  end


In [18]:
module Datum : DATUM = Datum

module Datum : DATUM


In [19]:
Datum.naredi 1991 6 25

- : Datum.t option = Some {Datum.leto = 1991; mesec = 6; dan = 25}


In [20]:
Datum.dolzina_meseca 1991 6

error: compile_error

## Abstraktni tipi

In [22]:
Datum.to_string { leto = 2024; mesec = 13; dan = 32 }

- : string = "32. 13. 2024"


In [23]:
module type DATUM =
    sig
      type t
      val naredi : int -> int -> int -> t option
      val to_string : t -> string
    end

module type DATUM =
  sig
    type t
    val naredi : int -> int -> int -> t option
    val to_string : t -> string
  end


In [24]:
module Datum : DATUM = Datum

module Datum : DATUM


In [27]:
type datum = { leto : int; mesec : int; dan : int }


type datum = { leto : int; mesec : int; dan : int; }


In [28]:
Datum.to_string  { leto = 2024; mesec = 13; dan = 32 }

error: compile_error

## Štetje različnih elementov

In [1]:
let stevilo_razlicnih xs =
    let rec aux ze_videni = function
      | [] -> List.length ze_videni
      | x :: xs' ->
          if List.mem x ze_videni
          then aux ze_videni xs'
          else aux (x :: ze_videni) xs'
    in
    aux [] xs

val stevilo_razlicnih : 'a list -> int = <fun>


In [2]:
stevilo_razlicnih [1; 2; 1; 2; 1; 2; 3]

- : int = 3


In [3]:
let nakljucni_seznam m n = List.init n (fun _ -> Random.int m)

val nakljucni_seznam : int -> int -> int list = <fun>


In [4]:
nakljucni_seznam 5 20

- : int list = [4; 0; 2; 1; 4; 0; 4; 0; 1; 0; 2; 0; 4; 3; 0; 2; 1; 1; 3; 4]


In [5]:
stevilo_razlicnih @@ nakljucni_seznam 5 20

- : int = 5


In [7]:
let stopaj f x =
    let zacetek = Sys.time () in
    let y = f x in
    let konec = Sys.time () in
    let izpis =
      Printf.sprintf "Porabljen čas: %f ms" (1000. *. (konec -. zacetek))
    in
    print_endline izpis;
    y

val stopaj : ('a -> 'b) -> 'a -> 'b = <fun>


In [15]:
stopaj stevilo_razlicnih (nakljucni_seznam 1_000_000_000 2000)

Porabljen čas: 32.148000 ms


- : int = 2000


In [21]:
stopaj stevilo_razlicnih (nakljucni_seznam 1_000_000_000 4000)

Porabljen čas: 86.214000 ms


- : int = 4000


In [22]:
stopaj stevilo_razlicnih (nakljucni_seznam 1_000_000_000 8000)

Porabljen čas: 258.423000 ms


- : int = 8000


In [26]:
module type MNOZICA = sig
  type 'a t
  val dodaj : 'a t -> 'a -> 'a t
  val prazna : 'a t
  (* val odstrani : 'a t -> 'a -> 'a t *)
  val moc : 'a t -> int
end

module type MNOZICA =
  sig
    type 'a t
    val dodaj : 'a t -> 'a -> 'a t
    val prazna : 'a t
    val moc : 'a t -> int
  end


In [29]:
module Mnozica : MNOZICA = struct
  type 'a t = 'a list
  let dodaj mn x = if List.mem x mn then mn else x :: mn
  let prazna = []
  let moc mn = List.length mn
end

module Mnozica : MNOZICA


In [36]:
let stevilo_razlicnih xs =
    let rec aux ze_videni = function
      | [] -> Mnozica.moc ze_videni
      | x :: xs' -> aux (Mnozica.dodaj ze_videni x) xs'
    in
    aux Mnozica.prazna xs

val stevilo_razlicnih : 'a list -> int = <fun>


In [37]:
let seznam_zaporednih n = List.init n (fun i -> i + 1)

val seznam_zaporednih : int -> int list = <fun>


In [38]:
seznam_zaporednih 10

- : int list = [1; 2; 3; 4; 5; 6; 7; 8; 9; 10]


In [39]:
stevilo_razlicnih @@ seznam_zaporednih 10000

- : int = 10000


## Funktorji

In [46]:
module type KOLOBAR = sig
  type t

  val ( + ) : t -> t -> t
  val ( - ) : t -> t -> t
  val ( * ) : t -> t -> t
  val ( ! ) : int -> t
  val nic : t
  val ena : t
  val izpis : t -> string
end

module type KOLOBAR =
  sig
    type t
    val ( + ) : t -> t -> t
    val ( - ) : t -> t -> t
    val ( * ) : t -> t -> t
    val ( ! ) : int -> t
    val nic : t
    val ena : t
    val izpis : t -> string
  end


In [47]:
module CelaStevila : KOLOBAR = struct
  type t = int

  let ( + ) = ( + )
  let ( - ) = ( - )
  let ( * ) = ( * )
  let nic = 0
  let ena = 1
  let ( ! ) n = n
  let izpis n = string_of_int n
end

module CelaStevila : KOLOBAR


In [48]:
CelaStevila.(ena + ena |> izpis)

- : string = "2"


In [49]:
module RealnaStevila : KOLOBAR = struct
  type t = float

  let ( + ) = ( +. )
  let ( - ) = ( -. )
  let ( * ) = ( *. )
  let nic = 0.
  let ena = 1.
  let ( ! ) n = float_of_int n
  let izpis x = string_of_float x
end

module RealnaStevila : KOLOBAR


In [50]:
RealnaStevila.(ena + ena |> izpis)

- : string = "2."


In [32]:
module type POLINOM = sig
  type polinom
  type kolobar

  val vrednost : polinom -> kolobar -> kolobar
  (** [vrednost p x] po Hornerjevem algoritmu izračuna vrednost polinoma [p] v
      točki [x]. *)

  val odvod : polinom -> polinom
  val izpis : polinom -> string
  val ( + ) : polinom -> polinom -> polinom
  val ( * ) : polinom -> polinom -> polinom
  val x : polinom
  val ( !! ) : int -> polinom
  val ( ! ) : kolobar -> polinom
  val ( ** ) : polinom -> int -> polinom
  val iz_koeficientov : kolobar list -> polinom
  val v_koeficiente : polinom -> kolobar list
end

module type POLINOM =
  sig
    type polinom
    type kolobar
    val vrednost : polinom -> kolobar -> kolobar
    val odvod : polinom -> polinom
    val izpis : polinom -> string
    val ( + ) : polinom -> polinom -> polinom
    val ( * ) : polinom -> polinom -> polinom
    val x : polinom
    val ( !! ) : int -> polinom
    val ( ! ) : kolobar -> polinom
    val ( ** ) : polinom -> int -> polinom
    val iz_koeficientov : kolobar list -> polinom
    val v_koeficiente : polinom -> kolobar list
  end


In [ ]:
module Polinom (K : KOLOBAR) : POLINOM with type kolobar = K.elt = struct
  type polinom = K.t list
  type kolobar = K.t

  let stevke b n =
    let rec aux acc n = if n = 0 then acc else aux ((n mod b) :: acc) (n / b) in
    aux [] n

  let filter_mapi f xs =
    let rec aux i = function
      | [] -> []
      | x :: xs -> (
          match f i x with
          | None -> aux (i + 1) xs
          | Some y -> y :: aux (i + 1) xs)
    in
    aux 0 xs

  let pocisti p =
    List.fold_right
      (fun x acc ->
        match (x, acc) with a, [] when a = K.nic -> [] | _ -> x :: acc)
      p []

  let sestej p1 p2 =
    let rec aux = function
      | [], p2 -> p2
      | p1, [] -> p1
      | a1 :: p1, a2 :: p2 -> K.(a1 + a2) :: aux (p1, p2)
    in
    aux (p1, p2) |> pocisti

  let zmnozi p1 p2 =
    p1
    |> List.mapi (fun k a ->
        List.init k (fun _ -> K.nic) @ List.map (K.( * ) a) p2)
    |> List.fold_left sestej []

  let rec potenca p =
    (* uporabimo hitro potenciranje *)
    function
    | 0 -> [ K.ena ] (* p⁰ = 1 *)
    | n ->
        let p2 = zmnozi p p in
        if n mod 2 = 0 then potenca p2 (n / 2) (* p²ⁿ = (p²)ⁿ *)
        else zmnozi (potenca p2 (n / 2)) p (* p²ⁿ⁺¹ = (p²)ⁿ ⋅ p *)

  let vrednost p x = List.fold_right (fun a v -> K.((v * x) + a)) p K.nic

  let odvod = function
    | [] -> []
    | _ :: p -> List.mapi (fun k a -> K.(a * (!k + !1))) p

  let izpis p =
    let eksponent k =
      stevke 10 k
      |> List.map
           (Array.get [| "⁰"; "¹"; "²"; "³"; "⁴"; "⁵"; "⁶"; "⁷"; "⁸"; "⁹" |])
      |> String.concat ""
    in
    let clen k a =
      let monom =
        match k with
        | 0 -> K.izpis a
        | 1 when a = K.ena -> "x"
        | 1 -> K.izpis a ^ " x"
        | k when a = K.ena -> "x" ^ eksponent k
        | k -> K.izpis a ^ " x" ^ eksponent k
      in
      if a = K.nic then None else Some monom
    in
    match p |> filter_mapi clen |> List.rev with
    | [] -> K.izpis K.nic
    | cleni -> String.concat " + " cleni

  let ( + ) = sestej
  let ( * ) = zmnozi
  let x = [ K.nic; K.ena ]
  let ( ** ) = potenca
  let iz_koeficientov p = pocisti p
  let v_koeficiente p = p
  let ( ! ) a = pocisti [ a ]
  let ( !! ) n = pocisti [ K.( ! ) n ]
end

error: compile_error

In [35]:
module CelostevilskiPolinom = Polinom (CelaStevila)
module RealniPolinom = Polinom (RealnaStevila)

error: compile_error

In [ ]:
CelostevilskiPolinom.((x + !!1) ** 3 |> odvod |> izpis)

In [ ]:
RealniPolinom.((x + !!1) ** 3 |> odvod |> izpis)

In [ ]:
CelostevilskiPolinom.(vrednost ((x + !!1) ** 3) 2)

In [ ]:
RealniPolinom.(vrednost ((x + !!1) ** 3) 0.5)

In [ ]:
CelostevilskiPolinom.((x + !!(-1)) ** 3 |> izpis)